<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Previsão de Demência**
---


🎯 **Objetivo:**  Prever sinais de demência, através de informações clínicas e demográficas de pacientes com potencial risco de Alzheimer (OASIS).


---


Desafio Estatística com Python - Classificação

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Subject ID`: Identificador único do paciente
- `MRI ID`: Identificador único do exame
- `Group` (alvo): Classificação do paciente
  - `Nondemented` - será tratada para variável binária 0
  - `Demented` e `Converted` - serão tratadas para variável binária 1
- `Visit`: Identificador da visita de cada paciente
- `MR Delay`: Intervalo em dias entre os exames
- `M/F`: Gênero (M: masculino, F: feminino)
- `Hand`: Mão dominante
- `Age`: Idade do paciente (numérico)
- `EDUC`: Anos de escolaridade (numérico)
- `SES`: Status socioeconômico (1 a 5)
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30)
- `CDR`: Clinical Dementia Rating (0 a 3)
- `eTIV`: Volume intracraniano estimado
- `nWBV`: Proporção de volume cerebral normalizado
- `ASF`: Fator de escala anatômica

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from IPython.display import display, Markdown

# Carregamento da base de dados
arquivo = 'oasis_longitudinal'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/main/{arquivo}.csv'
df = pd.read_csv(url)

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

# data: dataframe contendo apenas as colunas de interesse
# nulos: colunas que possuem valores nulos
# var_features: seleção do df sem a variável alvo
# corr_rank: ranking de correlação com group
var_alvo = 'Group'

# Configurações visuais dos gráficos

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2)

##  2. Análise exploratória dos dados

In [ ]:
# Conhecendo os dados

n_cols = df.shape[1]
print(f'\nTotal de linhas: {df.shape[0]}')
print(f'Total de colunas: {n_cols}')
print('-' * 50)

# Verificando duplicatas

duplicados = df.duplicated().sum()
print(f'\nLinhas duplicadas na base: {duplicados}')

In [ ]:
# Verificando tipagem e nulos

info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})
print('\n--- Diagnóstico de Tipagem e Qualidade ---\n')
display(info_df)

nulos = df.columns[df.isna().any()].tolist()
# Armazenando colunas com valores nulos
print(f'Colunas com nulos identificadas: {nulos}')

# Grafico de nulos
msno.matrix(df, figsize=(10, 5), fontsize=9,
            color=sns.color_palette(paleta, n_colors=n_cols)[int(n_cols / 2)])
plt.title('Visualização de Valores Ausentes', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Conhecendo os dados

df.head()

In [ ]:
# Conhecendo os dados

df.describe()

In [ ]:
# Conhecendo os dados

df.describe(include=['object', 'string'])

In [ ]:
# Conhecendo os dados

df['Group'].unique()

In [ ]:
# Conhecendo os dados

df['M/F'].unique()

In [ ]:
# Conhecendo os dados

df['Hand'].unique()

In [ ]:
# Selecionando apenas colunas de interesse

data = df.drop(['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'], axis=1)
data.head()

In [ ]:
# Variável categórica: % de group

data['Group'].value_counts(normalize=True)

In [ ]:
# Variável categórica: tratamento de group

data['Group'] = data['Group'].map({'Demented': 1, 'Converted': 1, 'Nondemented': 0})
data.head()

In [ ]:
# Variável categórica: % de gênero

data['M/F'].value_counts(normalize=True)

In [ ]:
# Variável categórica: tratamento de gênero

data['M/F'] = data['M/F'].map({'M': 1, 'F': 0})
data.head()

In [ ]:
# Fazendo a seleção do df sem a variável alvo (para reutilização)

var_features = [col for col in data.columns if col != var_alvo]
var_features

In [ ]:
# Calculando a matriz de correlação

corr = data.corr()
corr

In [ ]:
# Exibindo heatmap

plt.figure(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', linewidth=0.5)
plt.title('Matriz de correlação', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# Analisando distribuição e desbalanceamento de Group

plt.figure(figsize=(8, 6))
ax = sns.countplot(data=data, x=var_alvo, palette=cores, hue=var_alvo, legend=False)
plt.title(f'Distribuição da Variável Alvo ({var_alvo})', fontsize=12, fontweight='bold')
plt.xlabel(f'{var_alvo}')
plt.ylabel('Contagem')

# Adicionando porcentagens nas barras
total = len(data)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{(height/total)*100:.1f}%',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', xytext=(0, 3),
                textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(var_features):
    sns.boxplot(data=data, x=var_alvo, y=var, ax=axes[i], palette=cores, hue=var_alvo, legend=False)
    axes[i].set_title(f'Boxplot: {var} por {var_alvo}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(var_features):
    sns.histplot(data=data, x=var, hue=var_alvo, kde=True, ax=axes[i], palette=cores, element='step')
    axes[i].set_title(f'Distribuição: {var}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Exibindo dispersões

sns.pairplot(data, hue=var_alvo, diag_kind='hist', palette=cores)

In [ ]:
# Ranqueando os principais preditores

corr_rank = corr['Group'].abs().drop('Group').sort_values(ascending=False)
corr_rank

In [ ]:
display(Markdown(
        f'Com base na análise exploratória e no ranqueamento de correlação linear, os principais candidatos a preditores da variável alvo {var_alvo} são:\n'
        f'- {corr_rank.index[0]} (|r| = {corr_rank.iloc[0]:.2f}), \n'
        f'- {corr_rank.index[1]} (|r| = {corr_rank.iloc[1]:.2f}) e \n'
        f'- {corr_rank.index[2]} (|r| = {corr_rank.iloc[2]:.2f}).\n\n'
        f'Adicionalmente, as variáveis {corr_rank.index[3]} (|r| = {corr_rank.iloc[3]:.2f}) e {corr_rank.index[4]} (|r| = {corr_rank.iloc[4]:.2f}) demonstram associação intermediária, enquanto as demais características ({', '.join(corr_rank.index[5:])}) apresentam fraca correlação linear direta (|r| < 0.10) com o diagnóstico do grupo.'
))

## 3. Modelo de ML

Optamos por separar 75% dos dados para treino e 25% para teste.

In [ ]:
def tratar_nulos(dados_treino, dados_teste=None, tendencia='media'):
  """
  Trata valores ausentes usando a moda, calculada apenas no treino
  e aplicada em treino e teste (evita vazamento de dados).

  Parametros:
    dados_treino (pd.DataFrame): conjunto de treino.
    dados_teste (pd.DataFrame, opcional): conjunto de teste.
    tendencia (str, opcional): 'media', 'mediana' ou 'moda'. Padrao: 'media'.

  Retorno: tupla (X_train, X_test) com nulos preenchidos.
  """
  colunas_com_nulo = dados_treino.columns[dados_treino.isna().any()].tolist()

  for col in colunas_com_nulo:
    if tendencia == 'mediana':
      valor_tendencia = dados_treino[col].median()
    elif tendencia == 'moda':
      valor_tendencia = dados_treino[col].mode()[0]
    else:
      valor_tendencia = dados_treino[col].mean()

    dados_treino[col] = dados_treino[col].fillna(valor_tendencia)

    if dados_teste is not None:
      dados_teste[col] = dados_teste[col].fillna(valor_tendencia)

  return dados_treino, dados_teste


def preprocessar_dados(dados, alvo, test_size=0.2, random_state=42):
  """
  Preprocessa o dataframe para modelagem:
  1. Separa variaveis preditoras (X) e alvo (y).
  2. Faz a separacao estratificada dos dados em treino e teste.
  3. Normaliza as variaveis usando StandardScaler (ajustado no treino).
  4. Reconstroi dfs normalizados com colunas e indices originais.

  Parametros:
    dados (pd.DataFrame): df de entrada, ja tratado.
    alvo (str): nome da coluna davariavel alvo.
    test_size (float, opcional): proporcao dos dados para o conjunto de teste. Padrao: 0.2.

  Retorno: tupla: (X_train_scaled, X_test_scaled, y_train, y_test)
  """

  # Separa as variaveis para treino e teste
  X = dados.drop(columns=[alvo])
  y = dados[alvo]

  X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
  )

  # Trata nulos: moda calculada SO no treino, aplicada nos dois
  X_train, X_test = tratar_nulos(X_train, X_test, 'moda')

  # Normaliza com StandardScaler pra nao vazar dado
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)    # fit + transform no treino
  X_test_scaled = scaler.transform(X_test)          # so transform no teste

  # Traz o df de volta com os nomes das colunas e indices originais
  X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
  X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

  return X_train_scaled, X_test_scaled, y_train, y_test

In [ ]:
X_train_scaled, X_test_scaled, y_train, y_test = preprocessar_dados(data.drop(['CDR'], axis=1), 'Group', 0.25)

print(f'Tamanho do conjunto de treino: {X_train_scaled.shape[0]} amostras')
print(f'Tamanho do conjunto de teste: {X_test_scaled.shape[0]} amostras')

## Modelagem Preditiva - Pergunta 2

Nesta etapa, instanciamos os três algoritmos solicitados (Regressão Logística, Árvore de Decisão e Random Forest).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Instanciando e treinando
log_reg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predições
y_pred_log = log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_log = accuracy_score(y_test, y_pred_log)
prec_log = precision_score(y_test, y_pred_log, pos_label=1)
rec_log = recall_score(y_test, y_pred_log, pos_label=1)
f1_log = f1_score(y_test, y_pred_log, pos_label=1)
auc_log = roc_auc_score(y_test, y_prob_log)

print("Regressão Logística treinada com sucesso!")

Treinamento e Cálculo de Métricas

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Instanciando e treinando
dt_tree = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt_tree.fit(X_train_scaled, y_train)

# Predições
y_pred_dt = dt_tree.predict(X_test_scaled)
y_prob_dt = dt_tree.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt, pos_label=1)
rec_dt = recall_score(y_test, y_pred_dt, pos_label=1)
f1_dt = f1_score(y_test, y_pred_dt, pos_label=1)
auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Árvore de Decisão treinada com sucesso!")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Instanciando e treinando
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
rf_clf.fit(X_train_scaled, y_train)

# Predições
y_pred_rf = rf_clf.predict(X_test_scaled)
y_prob_rf = rf_clf.predict_proba(X_test_scaled)[:, 1]

# Métricas
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, pos_label=1)
rec_rf = recall_score(y_test, y_pred_rf, pos_label=1)
f1_rf = f1_score(y_test, y_pred_rf, pos_label=1)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest treinado com sucesso!")

Exibição da Tabela Comparativa

In [ ]:
import pandas as pd
from IPython.display import display

dados_comparativos = {
    "Modelo": ["Regressão Logística", "Árvore de Decisão", "Random Forest"],
    "Acurácia": [acc_log, acc_dt, acc_rf],
    "Precisão": [prec_log, prec_dt, prec_rf],
    "Recall (Sensibilidade)": [rec_log, rec_dt, rec_rf],
    "F1-Score": [f1_log, f1_dt, f1_rf],
    "AUC-ROC": [auc_log, auc_dt, auc_rf]
}

df_comparativo = pd.DataFrame(dados_comparativos).set_index("Modelo")

tabela_estilizada = (
    df_comparativo.style
    # Define fundo branco e texto escuro para TODAS as células do corpo
    .set_properties(**{
        'background-color': '#ffffff',
        'color': '#212529',
        'padding': '8px',
        'text-align': 'center'
    })
    # Aplica a cor de destaque (rosa/coral) por cima nos maiores valores de cada coluna
    .highlight_max(axis=0, color='#fde2e4')
    .format({
        "Acurácia": "{:.1%}",
        "Precisão": "{:.1%}",
        "Recall (Sensibilidade)": "{:.1%}",
        "F1-Score": "{:.1%}",
        "AUC-ROC": "{:.3f}"
    })
    .set_caption("<b>Tabela Comparativa de Desempenho dos Modelos de ML</b>")
    .set_table_styles([
        # Título
        {'selector': 'caption', 'props': [
            ('font-size', '14px'), 
            ('text-align', 'center'), 
            ('margin-bottom', '10px'),
            ('color', '#ffffff'), # Texto do título em branco para se destacar no tema escuro
            ('font-weight', 'bold')
        ]},
        # Cabeçalho da tabela e coluna de índices
        {'selector': 'th', 'props': [
            ('background-color', '#f1f3f5'), 
            ('color', '#212529'), 
            ('text-align', 'center'),
            ('font-weight', 'bold'),
            ('padding', '8px')
        ]},
        # Borda da tabela
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('border', '1px solid #dee2e6')
        ]}
    ])
)

display(tabela_estilizada)

A tabela apresenta o comparativo de métricas dos três modelos testados (**Regressão Logística**, **Árvore de Decisão** e **Random Forest**). Como o objetivo é prever sinais de demência, a avaliação foca especialmente na **Classe 1 (Demented/Converted)** — pacientes identificados com declínio cognitivo — mantendo o equilíbrio em relação à **Classe 0 (Nondemented)**.

### Interpretação das Métricas Chave:

* **Recall (Sensibilidade) da Classe 1:** Métrica crítica no contexto da saúde, pois indica a capacidade de identificar corretamente os pacientes doentes e evitar falsos negativos.
  * A **Regressão Logística** obteve o maior Recall (**71,7%**), seguida pelo **Random Forest** (**67,4%**) e pela **Árvore de Decisão** (**58,7%**).

* **Precisão (Precision) da Classe 1:** Mede a confiabilidade de um diagnóstico positivo (evitar alarmes falsos em pacientes saudáveis).
  * O **Random Forest** destacou-se com a maior precisão (**86,1%**), seguido pela **Regressão Logística** (**78,6%**) e pela **Árvore de Decisão** (**77,1%**).

* **Acurácia Geral e AUC-ROC:**
  * O **Random Forest** alcançou a melhor **Acurácia Geral** (**78,7%**) e o maior **F1-Score** (**75,6%**).
  * A **Regressão Logística** obteve a melhor capacidade de separação global (**AUC-ROC de 0.883**).

---

## Avaliação dos Modelos

1. Regressão Logística

In [ ]:
# Métricas

print(classification_report(y_test, y_pred_log))
print('\n' + '-'*60 + '\n')

# Matriz de Confusão

matriz_conf_log = confusion_matrix(y_test, y_pred_log)
plt.figure()
sns.heatmap(matriz_conf_log, annot=True, fmt='d', cmap=paleta, cbar=False, annot_kws={"size": 14})
plt.title('Matriz de Confusão - Regressão Logística', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n' + '-'*60 + '\n')

# Peso das variáveis

peso_log = pd.DataFrame({
    'Variável': X_train_scaled.columns,
    'Peso': log_reg.coef_[0]
}).sort_values(by='Peso', ascending=False)

plt.figure()
ax = sns.barplot(data=peso_log, x='Peso', y='Variável', hue='Variável', palette=paleta)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', label_type='edge', fontsize=10, weight='bold', color='black', padding=2)
plt.title('Peso das Variáveis - Regressão Logística', fontsize=14, fontweight='bold')
plt.xlabel('Peso', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

2. Árvore de Decisão

In [ ]:
# Métricas

print(classification_report(y_test, y_pred_dt))
print('\n' + '-'*60 + '\n')

# Matriz de Confusão

matriz_conf_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure()
sns.heatmap(matriz_conf_dt, annot=True, fmt='d', cmap=paleta, cbar=False, annot_kws={"size": 14})
plt.title('Matriz de Confusão - Árvore de Decisão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n' + '-'*60 + '\n')

# Importância das variáveis

importancia_dt = pd.DataFrame({
    'Variável': X_train_scaled.columns,
    'Importância Relativa': dt_tree.feature_importances_ 
}).sort_values(by='Importância Relativa', ascending=False)

plt.figure()
ax = sns.barplot(data=importancia_dt, x='Importância Relativa', y='Variável', hue='Variável', palette=paleta)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', fontsize=10, weight='bold', color='black', padding=2)
plt.title('Impacto das Variáveis - Árvore de Decisão', fontsize=14, fontweight='bold')
plt.xlabel('Importância Relativa', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

3. Random Forest

In [ ]:
# Métricas

print(classification_report(y_test, y_pred_rf))
print('\n' + '-'*60 + '\n')

# Matriz de Confusão

matriz_conf_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure()
sns.heatmap(matriz_conf_rf, annot=True, fmt='d', cmap=paleta, cbar=False, annot_kws={"size": 14})
plt.title('Matriz de Confusão - Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n' + '-'*60 + '\n')

# Importância das variáveis

importancia_rf = pd.DataFrame({
    'Variável': X_train_scaled.columns,
    'Importância Relativa': rf_clf.feature_importances_ 
}).sort_values(by='Importância Relativa', ascending=False)

plt.figure()
ax = sns.barplot(data=importancia_rf, x='Importância Relativa', y='Variável', hue='Variável', palette=paleta)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', fontsize=10, weight='bold', color='black', padding=2)
plt.title('Impacto das Variáveis - Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importância Relativa', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

Para fazer a seleção do melhor modelo, devemos considerar algumas coisas além de somente a acurácia geral.
Por exemplo, o F1 score, que avalia se o algoritmo consegue ser assertivo, o equilíbrio entre a Precisão e o Recall. E, também, o Recall (Sensibilidade) com o foco de reduzir ao máximo os Falsos Negativos.

In [ ]:
print('Métricas para seleção do melhor modelo:')

print(f'Regressão Logística: Acurácia = {acc_log:.2%}, Recall = {rec_log:.2%}, F1-Score = {f1_log:.2%}')
print(f'Árvore de Decisão: Acurácia = {acc_dt:.2%}, Recall = {rec_dt:.2%}, F1-Score = {f1_dt:.2%}')
print(f'Random Forest: Acurácia = {acc_rf:.2%}, Recall = {rec_rf:.2%}, F1-Score = {f1_rf:.2%}')

---

Quando olhamos somente para a Acurácia geral, percebemos que a Random Forest apresenta a maior entre os três com 78,72%, mas como disse não devemos ter somente essa métrica como parâmetro de seleção. Na Regressão Logística, o Recall atingiu 71,74% contra 67,39% da Random Forest, demonstrando que o modelo de regressão foi mais eficiente (sensível) em captar pacientes com demência, o que acarretou em um menor número de falsos negativos. Já para o F1 score, percebe-se uma diferença de 0,61% entre esses dois modelos de classificação, demonstrando que ambos os modelos atingiram praticamente o mesmo nível de equilíbrio geral.
Portanto, o melhor modelo é o modelo de Regressão Logística